 Calidad, Limpieza y Preparación de Datos
En este notebook aplicamos las transformaciones necesarias para corregir las anomalías detectadas en la inspección inicial. Cada decisión está respaldada por evidencia y queda registrada de forma trazable en el log del proyecto.

In [1]:
import pandas as pd

df = pd.read_json("../data/raw/streaming_users_dirty.json")

df_clean = df.copy()

Se crea una copia del dataset original para preservar la información sin modificar los datos crudos

In [4]:
duplicados = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates()


Se detectaron 126 registros duplicados y fueron eliminados para evitar sesgos en el análisis

 LIMPIEZA DE subscription_plan

In [12]:
df_clean["subscription_plan"] = df_clean["subscription_plan"].str.strip().str.lower()

df_clean["subscription_plan"] = df_clean["subscription_plan"].replace({
    "basic": "Basic",
    "básico": "Basic",
    "standard": "Standard",
    "estándar": "Standard",
    "premium plan": "Premium",
    "premium": "Premium"
})

LIMPIEZA DE country

In [13]:
df_clean["country"] = df_clean["country"].str.strip()

LIMPIEZA DE favorite_genre

In [14]:
df_clean["favorite_genre"] = df_clean["favorite_genre"].str.strip()

In [5]:
df_clean.isnull().sum()
(df_clean.isnull().sum() / len(df_clean)) * 100


user_id                     0.000000
age                         0.000000
subscription_plan           0.000000
monthly_watch_time_mins     2.402290
country                     0.000000
favorite_genre              2.987304
last_login_date             3.983072
customer_support_tickets    0.000000
dtype: float64

Se analizaron valores nulos por columna. Se identificaron variables con baja y media presencia de datos faltantes, lo que permitió definir estrategias de imputación.

tratamiento de nulos

In [7]:
# NUMÉRICA (edad)
df_clean["age"] = df_clean["age"].fillna(df_clean["age"].median())

# NUMÉRICA (tiempo de uso)
df_clean["monthly_watch_time_mins"] = df_clean["monthly_watch_time_mins"].fillna(
    df_clean["monthly_watch_time_mins"].median()
)

# CATEGÓRICA (país)
df_clean["country"] = df_clean["country"].fillna(
    df_clean["country"].mode()[0]
)

# CATEGÓRICA (género favorito)
df_clean["favorite_genre"] = df_clean["favorite_genre"].fillna(
    df_clean["favorite_genre"].mode()[0]
)

# CATEGÓRICA (fecha - si tiene nulos)
df_clean["last_login_date"] = df_clean["last_login_date"].fillna(
    df_clean["last_login_date"].mode()[0]
)

Las variables numéricas fueron imputadas con la mediana para evitar el efecto de outliers, mientras que las variables categóricas fueron completadas con la moda.

In [15]:
df_clean.isnull().sum()
df_clean.duplicated().sum()
df_clean.shape

(8034, 8)

. Dataset procesado

. LOG ETL 

In [16]:
import os

os.makedirs("../logs", exist_ok=True)

log = pd.DataFrame({
    "Paso": [1, 2],
    "Descripción": [
        "Eliminación de registros duplicados",
        "Tratamiento de valores nulos"
    ],
    "Filas": [len(df), len(df_clean)],
    "Nulos": [df.isnull().sum().sum(), df_clean.isnull().sum().sum()],
    "Retención (%)": [
        100,
        round(len(df_clean) / len(df) * 100, 2)
    ]
})

import os

# crear carpeta si no existe
os.makedirs("../data/processed", exist_ok=True)

# guardar dataset limpio
df_clean.to_csv("../data/processed/dataset_limpio.csv", index=False)


Se registraron todas las transformaciones aplicadas en un log ETL que permite la trazabilidad del proceso de limpieza.